In [3]:
# imports
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm
import matplotlib.pyplot as plt
from sklearn.linear_model import Lasso, LogisticRegression, LassoCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, 
    mean_squared_error, 
    confusion_matrix, 
    ConfusionMatrixDisplay, 
    classification_report, 
    roc_curve, 
    roc_auc_score
)
from sklearn.utils import resample
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [4]:
# load all cleaned dfs
clean_2018 = pd.read_csv('clean_2018_rs.csv')
clean_2019 = pd.read_csv('clean_2019_rs.csv')
clean_2020 = pd.read_csv('clean_2020_rs.csv')
clean_2021 = pd.read_csv('clean_2021_rs.csv')
clean_2022 = pd.read_csv('clean_2022_rs.csv')

In [5]:
# function to rename variables in preparation for making it a dummy variable
def rename_variable(df):
    
    # add 000 to make it the baseline
    # change to make numbers into categorical strings
    # make dictionary 
    columns_to_rename = {
        'purchaser_type': {0: 'Not applicable', 1: 'Fannie Mae', 2: 'Ginnie Mae', 3: 'Freddie Mac', 4: 'Farmer Mac', 5: 'Private securitizer', 6: 'Commercial bank, savings bank, or savings association', 71: 'Credit union, mortgage company, or finance company', 72: 'Life insurance company', 8: 'Affiliate institution', 9: 'Other type of purchaser'},
        'preapproval': {1: 'Preapproval requested', 2: 'Preapproval not requested'},
        'loan_type': {1: 'Conventional (not insured or guaranteed by FHA, VA, RHS, or FSA)', 2: 'Federal Housing Administration insured (FHA)', 3: 'Veterans Affairs guaranteed (VA)', 4: 'USDA Rural Housing Service or Farm Service Agency guaranteed (RHS or FSA)'},
        'lien_status': {1: 'Secured by a first lien', 2: 'Secured by a subordinate lien'},
        'reverse_mortgage': {1: 'Reverse mortgage', 2: 'Not a reverse mortgage', 1111: 'Exempt'},
        'open-end_line_of_credit': {1: 'Open-end line of credit', 2: 'Not an open-end line of credit', 1111: 'Exempt'},
        'hoepa_status': {1: 'High-cost mortgage', 2: 'Not a high-cost mortgage', 3: 'Not applicable'},
        'negative_amortization': {1: 'Negative amortization', 2: 'No negative amortization', 1111: 'Exempt'},
        'interest_only_payment': {1: 'Interest-only payments', 2: 'No interest-only payments', 1111: 'Exempt'},
        'balloon_payment': {1: 'Balloon payment', 2: 'No balloon payment', 1111: 'Exempt'},
        'other_nonamortizing_features': {1: 'Other non-fully amortizing features', 2: 'No other non-fully amortizing features', 1111: 'Exempt'},
        'construction_method': {1: 'Site-built', 2: 'Manufactured home'},
        'manufactured_home_secured_property_type': {1: 'Manufactured home and land', 2: 'Manufactured home and not land', 3: 'Not applicable', 1111: 'Exempt'},
        'manufactured_home_land_property_interest': {1: 'Direct ownership', 2: 'Indirect ownership', 3: 'Paid leasehold', 4: 'Unpaid leasehold', 5: 'Not applicable', 1111: 'Exempt'},
        'total_units': {1: '1', 2: '2', 3: '3', 4: '4'},
        'applicant_credit_score_type': {1: 'Equifax Beacon 5.0', 2: 'Experian Fair Isaac', 3: 'FICO Risk Score Classic 04', 4: 'FICO Risk Score Classic 98', 5: 'VantageScore 2.0', 6: 'VantageScore 3.0', 7: 'More than one credit scoring model', 8: 'Other credit scoring model', 9: 'Not applicable', 1111: 'Exempt'},
        'co-applicant_credit_score_type': {1: 'Equifax Beacon 5.0', 2: 'Experian Fair Isaac', 3: 'FICO Risk Score Classic 04', 4: 'FICO Risk Score Classic 98', 5: 'VantageScore 2.0', 6: 'VantageScore 3.0', 7: 'More than one credit scoring model', 8: 'Other credit scoring model', 9: 'Not applicable', 10: 'No co-applicant', 1111: 'Exempt'},
        'applicant_ethnicity-1': {1: 'Hispanic or Latino', 2: '000 Not Hispanic or Latino', 3: 'Information not provided by applicant in mail, internet, or telephone application', 4: 'Not applicable'},
        'applicant_ethnicity-2': {1: 'Hispanic or Latino', 2: '000 Not Hispanic or Latino'},
        'applicant_ethnicity-3': {1: 'Hispanic or Latino', 2: '000 Not Hispanic or Latino'},
        'applicant_ethnicity-4': {1: 'Hispanic or Latino', 2: '000 Not Hispanic or Latino'},
        'applicant_ethnicity-5': {1: 'Hispanic or Latino', 2: '000 Not Hispanic or Latino'},
        'co-applicant_ethnicity-1': {1: 'Hispanic or Latino', 2: '000 Not Hispanic or Latino', 3: 'Information not provided by applicant in mail, internet, or telephone application', 4: 'Not applicable', 5: 'No co-applicant'},
        'co-applicant_ethnicity-2': {1: 'Hispanic or Latino', 2: '000 Not Hispanic or Latino'},
        'co-applicant_ethnicity-3': {1: 'Hispanic or Latino', 2: '000 Not Hispanic or Latino'},
        'co-applicant_ethnicity-4': {1: 'Hispanic or Latino', 2: '000 Not Hispanic or Latino'},
        'co-applicant_ethnicity-5': {1: 'Hispanic or Latino', 2: '000 Not Hispanic or Latino'},
        'applicant_ethnicity_observed': {1: 'Collected on the basis of visual observation or surname', 2: 'Not collected on the basis of visual observation or surname', 3: 'Not applicable'},
        'co-applicant_ethnicity_observed': {1: 'Collected on the basis of visual observation or surname', 2: 'Not collected on the basis of visual observation or surname', 3: 'Not applicable', 4: 'No co-applicant'},
        'applicant_race-1': {1: 'American Indian or Alaska Native', 2: 'Asian', 3: 'Black or African American', 4: 'Native Hawaiian or Other Pacific Islander', 5: '000 White', 6: 'Information not provided by applicant in mail, internet, or telephone application', 7: 'Not applicable'},
        'applicant_race-2': {1: 'American Indian or Alaska Native', 2: 'Asian', 3: 'Black or African American', 4: 'Native Hawaiian or Other Pacific Islander', 5: '000 White'},
        'applicant_race-3': {1: 'American Indian or Alaska Native', 2: 'Asian', 3: 'Black or African American', 4: 'Native Hawaiian or Other Pacific Islander', 5: '000 White'},
        'applicant_race-4': {1: 'American Indian or Alaska Native', 2: 'Asian', 3: 'Black or African American', 4: 'Native Hawaiian or Other Pacific Islander', 5: '000 White'},
        'applicant_race-5': {1: 'American Indian or Alaska Native', 2: 'Asian', 3: 'Black or African American', 4: 'Native Hawaiian or Other Pacific Islander', 5: '000 White'},
        'co-applicant_race-1': {1: 'American Indian or Alaska Native', 2: 'Asian', 3: 'Black or African American', 4: 'Native Hawaiian or Other Pacific Islander', 5: '000 White', 6: 'Information not provided by applicant in mail, internet, or telephone application', 7: 'Not applicable', 8: 'No co-applicant'},
        'co-applicant_race-2': {1: 'American Indian or Alaska Native', 2: 'Asian', 3: 'Black or African American', 4: 'Native Hawaiian or Other Pacific Islander', 5: '000 White'},
        'co-applicant_race-3': {1: 'American Indian or Alaska Native', 2: 'Asian', 3: 'Black or African American', 4: 'Native Hawaiian or Other Pacific Islander', 5: '000 White'},
        'co-applicant_race-4': {1: 'American Indian or Alaska Native', 2: 'Asian', 3: 'Black or African American', 4: 'Native Hawaiian or Other Pacific Islander', 5: '000 White'},
        'co-applicant_race-5': {1: 'American Indian or Alaska Native', 2: 'Asian', 3: 'Black or African American', 4: 'Native Hawaiian or Other Pacific Islander', 5: '000 White'},
        'applicant_race_observed': {1: 'Collected on the basis of visual observation or surname', 2: 'Not collected on the basis of visual observation or surname', 3: 'Not applicable'},
        'co-applicant_race_observed': {1: 'Collected on the basis of visual observation or surname', 2: 'Not collected on the basis of visual observation or surname', 3: 'Not applicable', 4: 'No co-applicant'},
        'applicant_sex': {1: '000 Male', 2: 'Female', 3: 'Information not provided by applicant in mail, internet, or telephone application', 4: 'Not applicable', 6: 'Applicant selected both male and female'},
        'co-applicant_sex': {1: '000 Male', 2: 'Female', 3: 'Information not provided by applicant in mail, internet, or telephone application', 4: 'Not applicable', 5: 'No co-applicant', 6: 'Co-applicant selected both male and female'},
        'applicant_sex_observed': {1: 'Collected on the basis of visual observation or surname', 2: 'Not collected on the basis of visual observation or surname', 3: 'Not applicable'},
        'co-applicant_sex_observed': {1: 'Collected on the basis of visual observation or surname', 2: 'Not collected on the basis of visual observation or surname', 3: 'Not applicable', 4: 'No co-applicant'},
        'submission_of_application': {1: 'Submitted directly to your institution', 2: 'Not submitted directly to your institution', 3: 'Not applicable', 1111: 'Exempt'},
        'initially_payable_to_institution': {1: 'Initially payable to your institution', 2: 'Not initially payable to your institution', 3: 'Not applicable', 1111: 'Exempt'},
        'aus-1': {1: 'Desktop Underwriter (DU)', 2: 'Loan Prospector (LP) or Loan Product Advisor', 3: 'Technology Open to Approved Lenders (TOTAL) Scorecard', 4: 'Guaranteed Underwriting System (GUS)', 5: 'Other', 6: 'Not applicable', 7: 'Internal Proprietary System', 1111: 'Exempt'},
        'aus-2': {1: 'Desktop Underwriter (DU)', 2: 'Loan Prospector (LP) or Loan Product Advisor', 3: 'Technology Open to Approved Lenders (TOTAL) Scorecard', 4: 'Guaranteed Underwriting System (GUS)', 5: 'Other', 6: 'Not applicable', 7: 'Internal Proprietary System'},
        'aus-3': {1: 'Desktop Underwriter (DU)', 2: 'Loan Prospector (LP) or Loan Product Advisor', 3: 'Technology Open to Approved Lenders (TOTAL) Scorecard', 4: 'Guaranteed Underwriting System (GUS)', 5: 'Other', 6: 'Not applicable', 7: 'Internal Proprietary System'},
        'aus-4': {1: 'Desktop Underwriter (DU)', 2: 'Loan Prospector (LP) or Loan Product Advisor', 3: 'Technology Open to Approved Lenders (TOTAL) Scorecard', 4: 'Guaranteed Underwriting System (GUS)', 5: 'Other', 6: 'Not applicable', 7: 'Internal Proprietary System'},
        'aus-5': {1: 'Desktop Underwriter (DU)', 2: 'Loan Prospector (LP) or Loan Product Advisor', 3: 'Technology Open to Approved Lenders (TOTAL) Scorecard', 4: 'Guaranteed Underwriting System (GUS)', 5: 'Other', 6: 'Not applicable', 7: 'Internal Proprietary System'},
        'derived_ethnicity': {'Not Hispanic or Latino': '000 Not Hispanic or Latino'},
        'derived_race': {'White': '000 White'},
        'derived_sex': {'Male': '000 Male'}}
    
    # apply dictionary
    for column, replacements in columns_to_rename.items():
        if column in df.columns:
            df[column] = df[column].replace(replacements)
    return df

In [6]:
# function that finds columns to based on alphabetic order and prefix 
def drop_first_alphabetically(bool_columns):
    
    # dictionary to group columns by prefix
    prefix_groups = {}

    # group columns by prefix
    for col in bool_columns:
        prefix = col.rsplit('_', 1)[0]
        if prefix not in prefix_groups:
            prefix_groups[prefix] = []
        prefix_groups[prefix].append(col)

    # list of columns to drop
    columns_to_drop = []

    # find which columns to drop based on alphabetic order
    for prefix, columns in prefix_groups.items():
        if len(columns) > 1:
            columns.sort()
            columns_to_drop.append(columns[0]) 
    return columns_to_drop

In [7]:
# function to create dummy variables
def dummy(df):
    
    # make categorical columns into dummies
    dummy_df = pd.get_dummies(df, drop_first=False)
    
    # drop columns to prevent collinearity 
    bool_columns = dummy_df.select_dtypes(include=bool).columns
    drop_columns = drop_first_alphabetically(bool_columns)
    keep_columns = list(set(bool_columns) - set(drop_columns))
    dummy_df = dummy_df.drop(columns=drop_columns) 
    dummy_df[keep_columns] = dummy_df[keep_columns].astype(int)
    return dummy_df

In [8]:
# function that combines previous functions
def save_dummy_df(df, year):
    dummy_df = dummy(rename_variable(df.dropna()))
    dummy_df.to_csv(f'dummy_{year}_rs.csv', index=False)

In [9]:
# apply combined function on cleaned df
save_dummy_df(clean_2018, '2018')
save_dummy_df(clean_2019, '2019')
save_dummy_df(clean_2020, '2020')
save_dummy_df(clean_2021, '2021')
save_dummy_df(clean_2022, '2022')

/var/folders/b9/kvwg4mzj5kv0208_l9z2ljsc0000gn/T/ipykernel_86847/434196771.py:67: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[column] = df[column].replace(replacements)
/var/folders/b9/kvwg4mzj5kv0208_l9z2ljsc0000gn/T/ipykernel_86847/434196771.py:67: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[column] = df[column].replace(replacements)
/var/folders/b9/kvwg4mzj5kv0208_l9z2ljsc0000gn/T/ipykernel_86847/434196771.py:67: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a D

## Examine whether the dropped values exhibit random or non-random patterns

In [10]:
#2018
clean_2018.describe()

,purchaser_type,preapproval,loan_type,lien_status,reverse_mortgage,open-end_line_of_credit,loan_amount,loan_to_value_ratio,interest_rate,rate_spread,...,submission_of_application,initially_payable_to_institution,aus-1,tract_population,tract_minority_population_percent,ffiec_msa_md_median_family_income,tract_to_msa_income_percentage,tract_owner_occupied_units,tract_one_to_four_family_homes,tract_median_age_of_housing_units
count,61903.000000,61903.000000,61903.000000,61903.000000,61903.000000,61903.000000,61903.000000,59971.000000,61577.000000,61286.000000,...,61903.000000,61903.000000,61903.000000,61903.000000,61903.000000,61903.000000,61903.000000,61903.000000,61903.000000,61903.000000
mean,14.425359,1.930052,1.383277,1.001260,1.999919,2.124437,197725.069867,90.028425,4.761382,0.739051,...,1.175694,1.094099,1.802740,5085.758186,31.042817,81418.189748,110.509895,1366.624089,1620.572169,44.265092
std,25.416981,0.255062,0.667833,0.035475,0.008987,11.792491,87645.770053,9.253894,0.402001,0.592800,...,4.475594,4.468932,11.849298,1648.195597,22.340951,3369.816002,31.655647,532.837962,630.219531,17.682353
min,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,5000.000000,55.311000,3.750000,-1.094000,...,1.000000,1.000000,1.000000,252.000000,0.110000,67900.000000,27.000000,17.000000,0.000000,0.000000
25%,1.000000,2.000000,1.000000,1.000000,2.000000,2.000000,135000.000000,80.000000,4.500000,0.277000,...,1.000000,1.000000,1.000000,3881.000000,13.820000,82400.000000,88.000000,985.000000,1203.000000,32.000000
50%,3.000000,2.000000,1.000000,1.000000,2.000000,2.000000,185000.000000,95.000000,4.750000,0.607000,...,1.000000,1.000000,1.000000,4934.000000,25.030000,82400.000000,109.000000,1340.000000,1611.000000,42.000000
75%,6.000000,2.000000,2.000000,1.000000,2.000000,2.000000,255000.000000,96.500000,5.000000,1.138000,...,1.000000,1.000000,2.000000,6224.000000,45.210000,82400.000000,131.000000,1691.000000,2031.000000,58.000000
max,72.000000,2.000000,4.000000,2.000000,2.000000,1111.000000,525000.000000,118.724000,5.750000,2.538000,...,1111.000000,1111.000000,1111.000000,10370.000000,93.590000,89000.000000,200.000000,2816.000000,3274.000000,76.000000


In [11]:
clean_2018.dropna().describe()

,purchaser_type,preapproval,loan_type,lien_status,reverse_mortgage,open-end_line_of_credit,loan_amount,loan_to_value_ratio,interest_rate,rate_spread,...,submission_of_application,initially_payable_to_institution,aus-1,tract_population,tract_minority_population_percent,ffiec_msa_md_median_family_income,tract_to_msa_income_percentage,tract_owner_occupied_units,tract_one_to_four_family_homes,tract_median_age_of_housing_units
count,56563.000000,56563.000000,56563.000000,56563.000000,56563.0,56563.0,56563.000000,56563.000000,56563.000000,56563.000000,...,56563.000000,56563.000000,56563.000000,56563.000000,56563.000000,56563.000000,56563.000000,56563.000000,56563.000000,56563.000000
mean,13.955501,1.934551,1.355338,1.000071,2.0,2.0,198856.407899,89.935602,4.754557,0.729222,...,1.176794,1.092852,1.647756,5079.456199,31.051817,81484.399696,110.742659,1363.080353,1612.764687,44.372841
std,24.989472,0.247319,0.647132,0.008409,0.0,0.0,87579.097564,9.253981,0.396377,0.586634,...,4.680715,4.674161,1.054187,1648.699314,22.237445,3344.280954,31.812057,533.939620,630.448819,17.737476
min,0.000000,1.000000,1.000000,1.000000,2.0,2.0,5000.000000,55.311000,3.750000,-1.094000,...,1.000000,1.000000,1.000000,252.000000,0.110000,67900.000000,27.000000,17.000000,0.000000,0.000000
25%,1.000000,2.000000,1.000000,1.000000,2.0,2.0,135000.000000,80.000000,4.500000,0.275000,...,1.000000,1.000000,1.000000,3874.000000,14.000000,82400.000000,88.000000,981.000000,1193.000000,32.000000
50%,3.000000,2.000000,1.000000,1.000000,2.0,2.0,185000.000000,95.000000,4.750000,0.596000,...,1.000000,1.000000,1.000000,4931.000000,25.160000,82400.000000,109.000000,1336.000000,1603.000000,42.000000
75%,6.000000,2.000000,2.000000,1.000000,2.0,2.0,255000.000000,96.500000,5.000000,1.116000,...,1.000000,1.000000,2.000000,6222.000000,45.120000,82400.000000,131.000000,1687.000000,2017.000000,58.000000
max,72.000000,2.000000,4.000000,2.000000,2.0,2.0,525000.000000,118.724000,5.750000,2.538000,...,1111.000000,1111.000000,6.000000,10370.000000,93.590000,89000.000000,200.000000,2816.000000,3274.000000,76.000000


In [12]:
#2019
clean_2019.describe()

,purchaser_type,preapproval,loan_type,lien_status,reverse_mortgage,open-end_line_of_credit,loan_amount,loan_to_value_ratio,interest_rate,rate_spread,...,submission_of_application,initially_payable_to_institution,aus-1,tract_population,tract_minority_population_percent,ffiec_msa_md_median_family_income,tract_to_msa_income_percentage,tract_owner_occupied_units,tract_one_to_four_family_homes,tract_median_age_of_housing_units
count,47274.000000,47274.000000,47274.000000,47274.000000,47274.000000,47274.000000,47274.000000,45357.000000,47261.000000,46848.000000,...,47274.00000,47274.000000,47274.00000,47274.000000,47274.000000,47274.000000,47274.000000,47274.000000,47274.000000,47274.000000
mean,13.840779,1.957017,1.342514,1.001142,1.999915,2.138808,217738.503194,89.837576,4.244526,0.672798,...,1.17094,1.060012,1.72725,5043.716356,33.889568,81909.176291,113.469603,1338.393007,1542.292000,44.667936
std,24.795392,0.202822,0.619083,0.033779,0.009198,12.493281,87661.944130,9.180527,0.491014,0.602030,...,0.37646,0.237511,1.15955,1596.602654,22.327843,253.793978,33.294479,520.417503,627.609627,17.916538
min,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,5000.000000,55.252000,2.750000,-1.160000,...,1.00000,1.000000,1.00000,252.000000,0.110000,81200.000000,21.000000,17.000000,0.000000,0.000000
25%,1.000000,2.000000,1.000000,1.000000,2.000000,2.000000,155000.000000,80.000000,3.875000,0.228000,...,1.00000,1.000000,1.00000,3881.000000,16.050000,82000.000000,90.000000,968.000000,1128.000000,32.000000
50%,3.000000,2.000000,1.000000,1.000000,2.000000,2.000000,205000.000000,95.000000,4.250000,0.542000,...,1.00000,1.000000,1.00000,4965.000000,28.610000,82000.000000,111.000000,1314.000000,1530.000000,43.000000
75%,6.000000,2.000000,2.000000,1.000000,2.000000,2.000000,275000.000000,96.500000,4.500000,1.051000,...,1.00000,1.000000,2.00000,6130.000000,48.710000,82000.000000,134.000000,1662.000000,1945.000000,58.000000
max,72.000000,2.000000,4.000000,2.000000,2.000000,1111.000000,505000.000000,117.603000,5.750000,2.507000,...,2.00000,2.000000,6.00000,9980.000000,93.590000,82000.000000,208.000000,2720.000000,3200.000000,76.000000


In [13]:
clean_2019.dropna().describe()

,purchaser_type,preapproval,loan_type,lien_status,reverse_mortgage,open-end_line_of_credit,loan_amount,loan_to_value_ratio,interest_rate,rate_spread,...,submission_of_application,initially_payable_to_institution,aus-1,tract_population,tract_minority_population_percent,ffiec_msa_md_median_family_income,tract_to_msa_income_percentage,tract_owner_occupied_units,tract_one_to_four_family_homes,tract_median_age_of_housing_units
count,43681.000000,43681.000000,43681.000000,43681.000000,43681.0,43681.0,43681.000000,43681.000000,43681.000000,43681.000000,...,43681.000000,43681.000000,43681.000000,43681.000000,43681.000000,43681.000000,43681.000000,43681.000000,43681.000000,43681.000000
mean,13.439734,1.957922,1.315721,1.000023,2.0,2.0,218917.721664,89.820716,4.235471,0.666012,...,1.180239,1.061583,1.619262,5035.716742,33.836158,81914.800485,113.789451,1335.196584,1533.844967,44.742863
std,24.364232,0.200769,0.592051,0.004785,0.0,0.0,87722.879445,9.152796,0.483046,0.594509,...,0.384390,0.240399,0.931477,1596.431290,22.225896,246.783404,33.415537,520.761514,626.720712,17.967543
min,0.000000,1.000000,1.000000,1.000000,2.0,2.0,5000.000000,55.263000,2.750000,-1.091000,...,1.000000,1.000000,1.000000,252.000000,0.110000,81200.000000,21.000000,17.000000,0.000000,0.000000
25%,1.000000,2.000000,1.000000,1.000000,2.0,2.0,155000.000000,80.000000,3.875000,0.229000,...,1.000000,1.000000,1.000000,3874.000000,16.050000,82000.000000,90.000000,966.000000,1118.000000,32.000000
50%,3.000000,2.000000,1.000000,1.000000,2.0,2.0,215000.000000,95.000000,4.250000,0.537000,...,1.000000,1.000000,1.000000,4938.000000,28.610000,82000.000000,112.000000,1307.000000,1524.000000,43.000000
75%,6.000000,2.000000,2.000000,1.000000,2.0,2.0,275000.000000,96.500000,4.500000,1.031000,...,1.000000,1.000000,2.000000,6129.000000,48.310000,82000.000000,135.000000,1660.000000,1941.000000,58.000000
max,72.000000,2.000000,4.000000,2.000000,2.0,2.0,505000.000000,117.603000,5.750000,2.507000,...,2.000000,2.000000,6.000000,9980.000000,93.590000,82000.000000,208.000000,2720.000000,3200.000000,76.000000


In [14]:
#2020
clean_2020.describe()

,purchaser_type,preapproval,loan_type,lien_status,reverse_mortgage,open-end_line_of_credit,loan_amount,loan_to_value_ratio,interest_rate,rate_spread,...,submission_of_application,initially_payable_to_institution,aus-1,tract_population,tract_minority_population_percent,ffiec_msa_md_median_family_income,tract_to_msa_income_percentage,tract_owner_occupied_units,tract_one_to_four_family_homes,tract_median_age_of_housing_units
count,53985.000000,53985.000000,53985.000000,53985.000000,53985.000000,53985.000000,53985.000000,53104.000000,53704.000000,53620.000000,...,53985.000000,53985.000000,53985.000000,53985.000000,53985.000000,53985.000000,53985.000000,53985.000000,53985.000000,53985.000000
mean,9.988349,1.964972,1.309216,1.004427,1.999889,2.019116,228784.940261,89.937183,3.176534,0.379668,...,1.125090,1.045439,1.581754,5042.884950,30.162282,84443.171251,115.898342,1370.096545,1584.344948,43.715995
std,21.234273,0.183853,0.621996,0.066390,0.010542,4.773194,94970.525015,8.846105,0.421066,0.500558,...,0.330824,0.208266,0.945905,1615.628388,19.959571,997.348154,32.779918,525.014124,629.745691,17.884220
min,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,5000.000000,55.263000,2.000000,-1.175000,...,1.000000,1.000000,1.000000,447.000000,0.110000,80200.000000,27.000000,11.000000,0.000000,0.000000
25%,1.000000,2.000000,1.000000,1.000000,2.000000,2.000000,155000.000000,80.000000,2.875000,0.021000,...,1.000000,1.000000,1.000000,3869.000000,14.550000,84800.000000,93.000000,1003.000000,1168.000000,31.000000
50%,3.000000,2.000000,1.000000,1.000000,2.000000,2.000000,225000.000000,95.000000,3.125000,0.279000,...,1.000000,1.000000,1.000000,4889.000000,24.940000,84800.000000,113.000000,1340.000000,1562.000000,42.000000
75%,6.000000,2.000000,1.000000,1.000000,2.000000,2.000000,285000.000000,96.500000,3.500000,0.689000,...,1.000000,1.000000,2.000000,6135.000000,43.280000,84800.000000,137.000000,1687.000000,1990.000000,57.000000
max,72.000000,2.000000,4.000000,2.000000,2.000000,1111.000000,545000.000000,120.723000,4.425000,2.010000,...,2.000000,2.000000,6.000000,10370.000000,86.100000,85200.000000,208.000000,2760.000000,3200.000000,76.000000


In [15]:
clean_2020.dropna().describe()

,purchaser_type,preapproval,loan_type,lien_status,reverse_mortgage,open-end_line_of_credit,loan_amount,loan_to_value_ratio,interest_rate,rate_spread,...,submission_of_application,initially_payable_to_institution,aus-1,tract_population,tract_minority_population_percent,ffiec_msa_md_median_family_income,tract_to_msa_income_percentage,tract_owner_occupied_units,tract_one_to_four_family_homes,tract_median_age_of_housing_units
count,51126.000000,51126.000000,51126.000000,51126.000000,51126.0,51126.0,51126.000000,51126.000000,51126.000000,51126.000000,...,51126.000000,51126.000000,51126.000000,51126.000000,51126.000000,51126.000000,51126.000000,51126.000000,51126.000000,51126.000000
mean,9.657024,1.967101,1.287955,1.000078,2.0,2.0,229645.581505,89.916896,3.178888,0.386268,...,1.128956,1.045926,1.568008,5043.515178,30.166892,84452.560341,116.017349,1369.777413,1581.665161,43.719419
std,20.872149,0.178374,0.595217,0.008845,0.0,0.0,93925.187723,8.803838,0.419442,0.498550,...,0.335154,0.209326,0.893717,1616.060797,19.934309,993.745525,32.840518,525.565296,629.868978,17.884837
min,0.000000,1.000000,1.000000,1.000000,2.0,2.0,5000.000000,55.263000,2.000000,-1.175000,...,1.000000,1.000000,1.000000,625.000000,0.110000,80200.000000,27.000000,11.000000,0.000000,0.000000
25%,1.000000,2.000000,1.000000,1.000000,2.0,2.0,155000.000000,80.000000,2.875000,0.028000,...,1.000000,1.000000,1.000000,3869.000000,14.680000,84800.000000,93.000000,1002.000000,1163.000000,31.000000
50%,3.000000,2.000000,1.000000,1.000000,2.0,2.0,225000.000000,95.000000,3.125000,0.285355,...,1.000000,1.000000,1.000000,4895.000000,24.940000,84800.000000,113.000000,1340.000000,1561.000000,42.000000
75%,6.000000,2.000000,1.000000,1.000000,2.0,2.0,285000.000000,96.500000,3.500000,0.695000,...,1.000000,1.000000,2.000000,6151.000000,43.280000,84800.000000,137.000000,1687.000000,1985.000000,58.000000
max,72.000000,2.000000,4.000000,2.000000,2.0,2.0,545000.000000,120.723000,4.425000,2.008000,...,2.000000,2.000000,6.000000,10370.000000,86.100000,85200.000000,208.000000,2760.000000,3200.000000,76.000000


In [16]:
#2021
clean_2021.describe()

,purchaser_type,preapproval,loan_type,lien_status,reverse_mortgage,open-end_line_of_credit,loan_amount,loan_to_value_ratio,interest_rate,rate_spread,...,submission_of_application,initially_payable_to_institution,aus-1,tract_population,tract_minority_population_percent,ffiec_msa_md_median_family_income,tract_to_msa_income_percentage,tract_owner_occupied_units,tract_one_to_four_family_homes,tract_median_age_of_housing_units
count,62799.000000,62799.000000,62799.000000,62799.000000,62799.000000,62799.000000,62799.000000,61995.000000,62726.000000,62434.000000,...,62799.000000,62799.000000,62799.000000,62799.000000,62799.000000,62799.000000,62799.000000,62799.000000,62799.000000,62799.000000
mean,11.947929,1.956846,1.259749,1.001481,1.999936,1.998869,250076.673195,89.206476,3.056566,0.443353,...,1.155831,1.054555,1.590551,4974.700982,31.061298,86651.067692,115.661587,1334.460469,1541.036099,44.477874
std,23.545838,0.203204,0.574590,0.038454,0.007981,0.033606,107170.794768,8.976475,0.300938,0.466231,...,0.362697,0.227111,4.521368,1587.974150,20.384937,1055.097064,34.334195,524.052298,637.252124,17.924321
min,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,5000.000000,55.263000,2.198000,-0.943000,...,1.000000,1.000000,1.000000,447.000000,0.110000,82600.000000,21.000000,30.000000,0.000000,0.000000
25%,1.000000,2.000000,1.000000,1.000000,2.000000,2.000000,165000.000000,80.000000,2.875000,0.118000,...,1.000000,1.000000,1.000000,3815.000000,14.990000,87100.000000,92.000000,962.000000,1115.000000,32.000000
50%,3.000000,2.000000,1.000000,1.000000,2.000000,2.000000,235000.000000,94.953000,3.000000,0.344000,...,1.000000,1.000000,1.000000,4852.000000,25.650000,87100.000000,112.000000,1314.000000,1539.000000,42.000000
75%,6.000000,2.000000,1.000000,1.000000,2.000000,2.000000,315000.000000,96.500000,3.250000,0.698000,...,1.000000,1.000000,2.000000,6090.000000,44.830000,87100.000000,138.000000,1661.000000,1957.000000,58.000000
max,72.000000,2.000000,4.000000,2.000000,2.000000,2.000000,615000.000000,107.790000,3.890000,1.947000,...,2.000000,2.000000,1111.000000,9980.000000,87.210000,87100.000000,213.000000,2760.000000,3200.000000,76.000000


In [17]:
clean_2021.dropna().describe()

,purchaser_type,preapproval,loan_type,lien_status,reverse_mortgage,open-end_line_of_credit,loan_amount,loan_to_value_ratio,interest_rate,rate_spread,...,submission_of_application,initially_payable_to_institution,aus-1,tract_population,tract_minority_population_percent,ffiec_msa_md_median_family_income,tract_to_msa_income_percentage,tract_owner_occupied_units,tract_one_to_four_family_homes,tract_median_age_of_housing_units
count,60340.000000,60340.000000,60340.000000,60340.000000,60340.0,60340.0,60340.000000,60340.000000,60340.000000,60340.000000,...,60340.000000,60340.000000,60340.000000,60340.000000,60340.000000,60340.000000,60340.000000,60340.000000,60340.000000,60340.000000
mean,11.887471,1.956944,1.241067,1.000017,2.0,2.0,250851.508121,89.199117,3.057132,0.445859,...,1.155055,1.053182,1.566821,4970.083610,31.122709,86668.569771,115.778936,1332.481273,1535.864982,44.535913
std,23.486018,0.202985,0.549350,0.004071,0.0,0.0,106792.941810,8.948431,0.298261,0.463386,...,0.361960,0.224398,0.893745,1587.907283,20.390353,1040.676566,34.357239,523.936202,635.705200,17.954081
min,0.000000,1.000000,1.000000,1.000000,2.0,2.0,15000.000000,55.263000,2.198000,-0.943000,...,1.000000,1.000000,1.000000,447.000000,0.110000,82600.000000,21.000000,30.000000,0.000000,0.000000
25%,1.000000,2.000000,1.000000,1.000000,2.0,2.0,165000.000000,80.000000,2.875000,0.122000,...,1.000000,1.000000,1.000000,3815.000000,15.040000,87100.000000,92.000000,962.000000,1112.000000,32.000000
50%,3.000000,2.000000,1.000000,1.000000,2.0,2.0,235000.000000,94.940000,3.000000,0.345950,...,1.000000,1.000000,1.000000,4841.000000,25.690000,87100.000000,112.000000,1307.000000,1529.000000,42.000000
75%,6.000000,2.000000,1.000000,1.000000,2.0,2.0,315000.000000,96.500000,3.250000,0.697000,...,1.000000,1.000000,2.000000,6090.000000,44.830000,87100.000000,138.000000,1660.000000,1941.000000,58.000000
max,72.000000,2.000000,4.000000,2.000000,2.0,2.0,615000.000000,107.790000,3.875000,1.947000,...,2.000000,2.000000,7.000000,9980.000000,87.210000,87100.000000,213.000000,2760.000000,3200.000000,76.000000


In [18]:
#2022
clean_2022.describe()

,purchaser_type,preapproval,loan_type,lien_status,reverse_mortgage,open-end_line_of_credit,loan_amount,loan_to_value_ratio,interest_rate,rate_spread,...,submission_of_application,initially_payable_to_institution,aus-1,tract_population,tract_minority_population_percent,ffiec_msa_md_median_family_income,tract_to_msa_income_percentage,tract_owner_occupied_units,tract_one_to_four_family_homes,tract_median_age_of_housing_units
count,77139.000000,77139.000000,77139.000000,77139.000000,77139.000000,77139.000000,77139.000000,75631.000000,77126.000000,76876.000000,...,77139.000000,77139.000000,77139.000000,77139.000000,77139.000000,77139.000000,77139.000000,77139.000000,77139.000000,77139.000000
mean,12.558135,1.941495,1.342252,1.001763,1.999961,1.997692,242437.612621,89.689218,5.078138,0.626670,...,1.161565,1.059399,1.676338,4790.558952,40.089734,104061.378810,104.847781,1274.336756,1536.070392,41.419399
std,24.177626,0.234697,0.616786,0.041952,0.006236,0.047982,107258.536559,9.400946,1.102357,0.614415,...,0.368054,0.236372,1.091607,1540.932720,25.673143,7536.993041,33.562626,499.894362,576.583656,20.292079
min,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,5000.000000,55.260000,1.750000,-1.120000,...,1.000000,1.000000,1.000000,667.000000,3.710000,84100.000000,13.000000,7.000000,16.000000,0.000000
25%,1.000000,2.000000,1.000000,1.000000,2.000000,2.000000,165000.000000,80.000000,4.175000,0.212000,...,1.000000,1.000000,1.000000,3694.000000,20.260000,105700.000000,81.000000,916.000000,1154.000000,27.000000
50%,3.000000,2.000000,1.000000,1.000000,2.000000,2.000000,225000.000000,95.000000,5.250000,0.575000,...,1.000000,1.000000,1.000000,4687.000000,33.220000,105700.000000,103.000000,1267.000000,1543.000000,44.000000
75%,6.000000,2.000000,2.000000,1.000000,2.000000,2.000000,305000.000000,96.500000,5.875000,1.031000,...,1.000000,1.000000,2.000000,5868.000000,55.660000,105700.000000,126.000000,1597.000000,1923.000000,58.000000
max,72.000000,2.000000,4.000000,2.000000,2.000000,2.000000,625000.000000,120.710000,8.500000,2.344000,...,2.000000,2.000000,7.000000,9050.000000,99.820000,118200.000000,199.000000,2572.000000,2990.000000,80.000000


In [19]:
clean_2022.describe()

,purchaser_type,preapproval,loan_type,lien_status,reverse_mortgage,open-end_line_of_credit,loan_amount,loan_to_value_ratio,interest_rate,rate_spread,...,submission_of_application,initially_payable_to_institution,aus-1,tract_population,tract_minority_population_percent,ffiec_msa_md_median_family_income,tract_to_msa_income_percentage,tract_owner_occupied_units,tract_one_to_four_family_homes,tract_median_age_of_housing_units
count,77139.000000,77139.000000,77139.000000,77139.000000,77139.000000,77139.000000,77139.000000,75631.000000,77126.000000,76876.000000,...,77139.000000,77139.000000,77139.000000,77139.000000,77139.000000,77139.000000,77139.000000,77139.000000,77139.000000,77139.000000
mean,12.558135,1.941495,1.342252,1.001763,1.999961,1.997692,242437.612621,89.689218,5.078138,0.626670,...,1.161565,1.059399,1.676338,4790.558952,40.089734,104061.378810,104.847781,1274.336756,1536.070392,41.419399
std,24.177626,0.234697,0.616786,0.041952,0.006236,0.047982,107258.536559,9.400946,1.102357,0.614415,...,0.368054,0.236372,1.091607,1540.932720,25.673143,7536.993041,33.562626,499.894362,576.583656,20.292079
min,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,5000.000000,55.260000,1.750000,-1.120000,...,1.000000,1.000000,1.000000,667.000000,3.710000,84100.000000,13.000000,7.000000,16.000000,0.000000
25%,1.000000,2.000000,1.000000,1.000000,2.000000,2.000000,165000.000000,80.000000,4.175000,0.212000,...,1.000000,1.000000,1.000000,3694.000000,20.260000,105700.000000,81.000000,916.000000,1154.000000,27.000000
50%,3.000000,2.000000,1.000000,1.000000,2.000000,2.000000,225000.000000,95.000000,5.250000,0.575000,...,1.000000,1.000000,1.000000,4687.000000,33.220000,105700.000000,103.000000,1267.000000,1543.000000,44.000000
75%,6.000000,2.000000,2.000000,1.000000,2.000000,2.000000,305000.000000,96.500000,5.875000,1.031000,...,1.000000,1.000000,2.000000,5868.000000,55.660000,105700.000000,126.000000,1597.000000,1923.000000,58.000000
max,72.000000,2.000000,4.000000,2.000000,2.000000,2.000000,625000.000000,120.710000,8.500000,2.344000,...,2.000000,2.000000,7.000000,9050.000000,99.820000,118200.000000,199.000000,2572.000000,2990.000000,80.000000
